# Scaling Behavior

## Imports

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

## Data

In [ ]:
models = [
    {"name": "Qwen3-Emb-8B", "million_parameters": 8000, "performance": (0.744, 0.769, 0.794), "color": "tab:blue", "marker": "X", "type": "Qwen3-Emb"},
    {"name": "Qwen3-Emb-4B", "million_parameters": 4000, "performance": (0.730, 0.759, 0.787), "color": "tab:blue", "marker": "X", "type": "Qwen3-Emb"},
    {"name": "Qwen3-Emb-0.6B", "million_parameters": 600, "performance": (0.697, 0.727, 0.758), "color": "tab:blue", "marker": "X", "type": "Qwen3-Emb"},

    {"name": "Qwen2-Emb-7B", "million_parameters": 7000, "performance": (0.712, 0.737, 0.761), "color": "tab:red", "marker": "o", "type": "Qwen2-Emb"},
    {"name": "Qwen2-Emb-1.5B", "million_parameters": 1500, "performance": (0.683, 0.710, 0.738), "color": "tab:red", "marker": "o", "type": "Qwen2-Emb"},

    {"name": "DeBERTaV3-large", "million_parameters": 434, "performance": (0.664, 0.692, 0.720), "color": "tab:green", "marker": "h", "type": "DeBERTa"},
    {"name": "DeBERTaV3-base", "million_parameters": 183, "performance": (0.660, 0.685, 0.709), "color": "tab:green", "marker": "h", "type": "DeBERTa"},

    {"name": "BERT-large", "million_parameters": 330, "performance": (0.667, 0.694, 0.722), "color": "tab:purple", "marker": "^", "type": "BERT"},
    {"name": "BERT-base", "million_parameters": 110, "performance": (0.674, 0.701, 0.727), "color": "tab:purple", "marker": "^", "type": "BERT"},

    {"name": "CLIMBR-T-Base", "million_parameters": 141, "performance": (0.746, 0.769, 0.792), "color": "tab:orange", "marker": "*", "type": "CLIMBR-T-Base"}
]

## Plotting

In [ ]:
# initialize the plot
fig, ax = plt.subplots(figsize=(8, 4), dpi=600)

# title and axis labels
ax.set_title("Clinical Prediction Performance vs Model Size", fontsize=16, fontweight="bold", pad=10)
ax.set_xlabel("Model Parameters (millions, log scale)", fontsize=12, fontweight="bold")
ax.set_ylabel("Macro AUROC (95% CI)", fontsize=12, fontweight="bold")

# set x and y scales
ax.set_xscale("log")
ax.set_xlim(75, 12000)
ax.set_ylim(0.625, 0.82)
ax.set_yticks(np.arange(0.65, 0.80, 0.025))

# set x ticks to match the main model sizes
xticks = [110, 183, 330, 600, 1500, 4000, 8000]
ax.set_xticks(xticks)
ax.set_xticklabels(xticks)

# add grid with logarithmic spacing
ax.grid(visible=True, which="both", axis="x", linestyle="-", alpha=0.2)
ax.grid(visible=True, which="major", axis="y", linestyle="-", alpha=0.2)

# dictionary to store model types and their colors
model_types = {}

# plot data points with confidence intervals
for model in models:
    x = model["million_parameters"]
    mean = model["performance"][1]
    lower = model["performance"][0]
    upper = model["performance"][2]
    color = model["color"]
    marker = model["marker"]
    model_type = model["type"]

    # plot the data points and confidence intervals
    ax.errorbar(
        x, mean, yerr=[[mean-lower], [upper-mean]],
        fmt=marker, color=color, markersize=9, alpha=1, 
        capsize=4, capthick=1.5  
    )
    
    # store unique colors for each model type
    if model_type not in model_types:
        model_types[model_type] = color

# horizontal dashed line for CLIMBR performance 
clmbr_performance = models[-1]["performance"][1] # mean performance of CLIMBR-T-Base
ax.axhline(y=clmbr_performance, color=model_types["CLIMBR-T-Base"], linestyle='--', linewidth=1, zorder=0)

# create legend elements
order = ["Qwen3-Emb", "Qwen2-Emb", "DeBERTa", "BERT", "CLIMBR-T-Base"]
markers = {"Qwen3-Emb": "X", "Qwen2-Emb": "o", "DeBERTa": "h", "BERT": "^", "CLIMBR-T-Base": "*"}
legend_elements = [
    plt.Line2D([0], [0], marker=markers[label], color=model_types[label],
               label=label, linestyle='None', markersize=9)
    for label in order
]

legend = ax.legend(
    handles=legend_elements,
    loc='center left',
    bbox_to_anchor=(1.02, 0.25),
    fontsize=10,
    title="Model Types",
    title_fontsize=10, 
    frameon=False,
    handletextpad=0.1,
    borderpad=0,
    labelspacing=0.4
)
legend._legend_box.align = "left"
legend.get_title().set_ha("left")
legend.get_title().set_position((6, 3)) # improve placing of legend title

# adjust layout to prevent legend cutoff
plt.tight_layout()

# save or display the plot
plt.savefig("../../EHRSHOT_ASSETS/figures_noel/performance_with_model_types.png", dpi=600, bbox_inches='tight') # based on figures_ben
plt.savefig("../../EHRSHOT_ASSETS/figures_noel/performance_with_model_types.pdf", dpi=600, bbox_inches='tight') 
plt.show()

# Multi-Panel

## Imports

In [ ]:
import re
import matplotlib.pyplot as plt
import numpy as np

## Configurations

In [ ]:
color_qwen3 = "tab:blue"
color_qwen = "tab:orange"
color_llama = "tab:green"
color_gmb = "tab:red"

marker_qwen3 = "X"
marker_qwen = "o"
marker_llama = "h" 
marker_gmb = "^"

## Utils

In [ ]:
def _plot_unified_legend(fig: plt.Figure, axes: plt.Axes, ncol: int | None = None, fontsize: int = 8) -> None:
    """
    Create a unified legend for the entire figure.
    
    Reference: plot._plot_unified_legend

    Args:
        fig (plt.Figure): The figure object.
        axes (plt.Axes): The axes object.
        ncol (int | None): The number of columns for the legend.
        fontsize (int): The fontsize for the legend.

    Returns:
        None
    """

    labels = []
    label2handle = {}
    for ax in axes.ravel():
        for h, l in zip(*ax.get_legend_handles_labels()):
            if l not in labels:
                labels.append(l)
                label2handle[l] = h
    legend_n_col: int = len([x for x in labels if '(Full)' not in x]) if ncol is None else ncol
    
    # create grouped legends by model type
    model_types = {'CLIMBR': [], 'LLM': [], 'BERT': []}
    for label in labels:
        if 'clmbr' in label.lower():
            model_types['CLIMBR'].append(label)
        elif any(llm in label.lower() for llm in ['llm', 'gpt', 'qwen']):
            model_types['LLM'].append(label)
        else:
            model_types['BERT'].append(label)
    
    # create legend with grouped models
    all_labels = []
    all_handles = []
    for model_type, type_labels in model_types.items():
        if type_labels:
            all_labels.extend(type_labels)
            all_handles.extend([label2handle[l] for l in type_labels])
    
    # calculate optimal number of columns (max 3 for readability)
    ncols = min(len(all_handles), 4)
    
    # let constrained_layout handle the legend positioning
    leg = fig.legend(all_handles, all_labels, loc='center', 
               ncol=ncols, fontsize=fontsize, frameon=False,
               bbox_to_anchor=(0.5, -0.045))

    # set legend title and label fontweight to normal
    if leg.get_title() is not None:
        leg.get_title().set_fontweight('normal')
    for text in leg.get_texts():
        text.set_fontweight('normal')

In [ ]:
def parse_data_string(data: str, reverse: bool = True) -> tuple[list[float], list[float], list[float]]:
    """
    Parse the data strings into means and confidence intervals.

    Args:
        data (str): The data string to parse.
        reverse (bool): Whether to reverse the order of the data.
        
    Returns:
        tuple: A tuple of means, lower, and upper.
    """

    matches = re.findall(r"([\d.]+) \(([\d.]+) - ([\d.]+)\)", data)
    matches = [m for m in reversed(matches)] if reverse else matches
    
    means = [float(m[0]) for m in matches]
    lower = [float(m[1]) for m in matches]
    upper = [float(m[2]) for m in matches]

    return means, lower, upper


## Setup

In [ ]:
plt.style.use("seaborn-v0_8-paper")
fig, axes = plt.subplots(1, 2, figsize=(8.27, 3), constrained_layout=True)

In [ ]:
plt.rcParams.update({
        'font.size': 8,
        'axes.titlesize': 10,
        'axes.labelsize': 8,
        'xtick.labelsize': 8,
        'ytick.labelsize': 8,
        'legend.fontsize': 8,
        'figure.dpi': 300,
        'lines.linewidth': 1.5,
        'axes.grid': True,
        'grid.alpha': 0.3
})

## Context Sizes

### Data

In [ ]:
qwen3_data = "0.769 (0.744 - 0.794) 0.772 (0.746 - 0.798) 0.768 (0.746 - 0.790) 0.759 (0.735 - 0.782) 0.741 (0.715 - 0.766)"
qwen_data = "0.737 (0.712 - 0.761) 0.760 (0.736 - 0.785) 0.772 (0.747 - 0.798) 0.760 (0.735 - 0.785) 0.748 (0.726 - 0.771)"
llama_data = "0.725 (0.701 - 0.750) 0.742 (0.713 - 0.770) 0.757 (0.732 - 0.783) 0.757 (0.732 - 0.782) 0.735 (0.708 - 0.762)"

context_sizes = [512, 1024, 2048, 4096, 8192]

In [ ]:
qwen3_means, qwen3_lower, qwen3_upper = parse_data_string(qwen3_data, reverse=True)
qwen_means, qwen_lower, qwen_upper = parse_data_string(qwen_data, reverse=True)
llama_means, llama_lower, llama_upper = parse_data_string(llama_data, reverse=True)

print("Check Qwen3 data:")
print(qwen3_means, qwen3_lower, qwen3_upper)

print("Check Qwen data:")
print(qwen_means, qwen_lower, qwen_upper)

print("Check Llama data:")
print(llama_means, llama_lower, llama_upper)

### Plotting (Creation)

In [ ]:
# initialize the plot
# plt.style.use("seaborn-v0_8-paper")
# fig, ax = plt.subplots(figsize=(6, 4), dpi=180)
ax = axes[0]

# title and axis labels
ax.set_title("Performance Across Context Sizes", fontsize=12, pad=10, fontweight="bold")
ax.set_xlabel("Input Context Size (log scale)", fontweight="bold")
ax.set_ylabel("Macro AUROC (95% CI)", fontweight="bold")

# logarithmic x-axis for context sizes
ax.set_xscale("log")
ax.minorticks_off()
ax.set_xticks(context_sizes)
ax.set_xticklabels(context_sizes, fontsize=10)
ax.set_ylim(0.59, 0.81)
ax.set_yticks(np.arange(0.60, 0.81, 0.05))

# improve grid
ax.grid(visible=True, which="both", linestyle="-", alpha=0.2)
ax.grid(True, which='minor', linestyle=':', alpha=0.2)

# format axes
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.tick_params(axis='both', which='major', labelsize=8)

# plot Qwen3 data
ax.plot(context_sizes, qwen3_means, label="Qwen3-Emb-8B",
        marker=marker_qwen3, linestyle="-", linewidth=1.5,
        markersize=8, color=color_qwen3,
        markeredgecolor="white", markeredgewidth=1.5, zorder=9)
ax.fill_between(context_sizes, qwen3_lower, qwen3_upper, color=color_qwen3, alpha=0.1, zorder=8)

# plot Qwen data
ax.plot(context_sizes, qwen_means, label="Qwen2-Emb-7B",
        marker=marker_qwen, linestyle="-", linewidth=1.5,
        markersize=8, color=color_qwen,
        markeredgecolor="white", markeredgewidth=1.5, zorder=7)
ax.fill_between(context_sizes, qwen_lower, qwen_upper, color=color_qwen, alpha=0.1, zorder=6)

# plot Llama data
ax.plot(context_sizes, llama_means, label="Llama3.1-LLM2Vec-8B",
        marker=marker_llama, linestyle="-", linewidth=1.5,
        markersize=8, color=color_llama,
        markeredgecolor="white", markeredgewidth=1.5, zorder=5)
ax.fill_between(context_sizes, llama_lower, llama_upper, color=color_llama, alpha=0.1, zorder=4)

# legend for model names
legend_elements = [
    plt.Line2D([0], [0], marker=marker_qwen3, color=color_qwen3, label="Qwen3-Emb-8B", linestyle="None", markersize=8),
    plt.Line2D([0], [0], marker=marker_qwen, color=color_qwen, label="Qwen2-Emb-7B", linestyle="None", markersize=8),
    plt.Line2D([0], [0], marker=marker_llama, color=color_llama, label="Llama3.1-LLM2Vec-8B", linestyle="None", markersize=8),
]
# ax.legend(handles=legend_elements, loc="lower left", frameon=False, fontsize=10, title="Model Names")
# _plot_unified_legend(fig, np.array([[ax]]), fontsize=9)

# adjust layout for better spacing
# plt.tight_layout()

# save the figure
# plt.savefig("../../EHRSHOT_ASSETS/figures_noel/context_size_performance_adapted.png", dpi=300, bbox_inches='tight')
# plt.savefig("../../EHRSHOT_ASSETS/figures_noel/context_size_performance_adapted.pdf", dpi=300, bbox_inches='tight')

# plt.show()

## Time Windows

### Data

In [ ]:
qwen3_data = "0.667 (0.642 - 0.692) 0.743 (0.721 - 0.766) 0.758 (0.735 - 0.780) 0.767 (0.742 - 0.792) 0.772 (0.747 - 0.797) 0.770 (0.745 - 0.795) 0.769 (0.744 - 0.794)"
qwen_data = "0.661 (0.637 - 0.685) 0.749 (0.721 - 0.777) 0.754 (0.729 - 0.778) 0.756 (0.734 - 0.779) 0.742 (0.718 - 0.766) 0.735 (0.709 - 0.761) 0.737 (0.712 - 0.761)"
llama_data = " 0.654 (0.628 - 0.680) 0.736 (0.706 - 0.766) 0.747 (0.718 - 0.776) 0.747 (0.716 - 0.777) 0.733 (0.704 - 0.761) 0.721 (0.699 - 0.744) 0.725 (0.701 - 0.750)"
gmb_data = "0.650 (0.622 - 0.678) 0.751 (0.726 - 0.775) 0.759 (0.735 - 0.782) 0.764 (0.741 - 0.787) 0.773 (0.748 - 0.799) 0.772 (0.745 - 0.800) 0.777 (0.756 - 0.799)"

time_windows = ["1 hour", "1 day", "1 week", "1 month", "1 year", "3 years", "full"]
time_windows_ticks = np.arange(1, len(time_windows) + 1)

In [ ]:
qwen3_means, qwen3_lower, qwen3_upper = parse_data_string(qwen3_data, reverse=False)
qwen_means, qwen_lower, qwen_upper = parse_data_string(qwen_data, reverse=False)
llama_means, llama_lower, llama_upper = parse_data_string(llama_data, reverse=False)
gmb_means, gmb_lower, gmb_upper = parse_data_string(gmb_data, reverse=False)

print("Check Qwen3 data:")
print(qwen3_means, qwen3_lower, qwen3_upper)

print("Check Qwen data:")
print(qwen_means, qwen_lower, qwen_upper)

print("Check Llama data:")
print(llama_means, llama_lower, llama_upper)

print("Check GMB data:")
print(gmb_means, gmb_lower, gmb_upper)

### Plotting (Creation)

In [ ]:
# initialize the plot
# plt.style.use("seaborn-v0_8-paper")
# fig, ax = plt.subplots(figsize=(6, 4), dpi=180)
ax = axes[1]

# clean plot: remove top and right spines
# ax.spines["top"].set_visible(False)
# ax.spines["right"].set_visible(False)

# title and axis labels
ax.set_title("Performance Across Time Windows", fontsize=12, pad=10, fontweight="bold")
ax.set_xlabel("Time Window", fontweight="bold")
ax.set_ylabel("Macro AUROC (95% CI)", fontweight="bold")

# logarithmic x-axis for context sizes
# ax.set_xscale("log")
ax.set_xticks(time_windows_ticks)
ax.set_xticklabels(time_windows, fontsize=10)
ax.set_ylim(0.59, 0.81) 
ax.set_yticks(np.arange(0.60, 0.81, 0.05))

# improve grid
ax.grid(visible=True, which="both", linestyle="-", alpha=0.2)
ax.grid(True, which='minor', linestyle=':', alpha=0.2)

# format axes
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.tick_params(axis='both', which='major', labelsize=8)

# plot Qwen3 data
ax.plot(time_windows_ticks, qwen3_means, label="Qwen3-Emb-8B",
        marker=marker_qwen3, linestyle="-", linewidth=1.5,
        markersize=8, color=color_qwen3,
        markeredgecolor="white", markeredgewidth=1.5, zorder=9)
ax.fill_between(time_windows_ticks, qwen3_lower, qwen3_upper, color=color_qwen3, alpha=0.1, zorder=8)

# plot Qwen data
ax.plot(time_windows_ticks, qwen_means, label="Qwen2-Emb-7B",
        marker=marker_qwen, linestyle="-", linewidth=1.5,
        markersize=8, color=color_qwen,
        markeredgecolor="white", markeredgewidth=1.5, zorder=7)
ax.fill_between(time_windows_ticks, qwen_lower, qwen_upper, color=color_qwen, alpha=0.1, zorder=6)

# plot Llama data
ax.plot(time_windows_ticks, llama_means, label="Llama3.1-LLM2Vec-8B",
        marker=marker_llama, linestyle="-", linewidth=1.5,
        markersize=8, color=color_llama,
        markeredgecolor="white", markeredgewidth=1.5, zorder=5)
ax.fill_between(time_windows_ticks, llama_lower, llama_upper, color=color_llama, alpha=0.1, zorder=4)

# plot GMB data
ax.plot(time_windows_ticks, gmb_means, label="Count-based + GBM",
        marker=marker_gmb, linestyle="-", linewidth=1.5,
        markersize=8, color=color_gmb,
        markeredgecolor="white", markeredgewidth=1.5, zorder=3)
ax.fill_between(time_windows_ticks, gmb_lower, gmb_upper, color=color_gmb, alpha=0.1, zorder=2)

# legend for model names
legend_elements = [
    plt.Line2D([0], [0], marker=marker_qwen3, color=color_qwen3, label="Qwen3-Emb-8B", linestyle="None", markersize=8),
    plt.Line2D([0], [0], marker=marker_qwen, color=color_qwen, label="Qwen2-Emb-7B", linestyle="None", markersize=8),
    plt.Line2D([0], [0], marker=marker_llama, color=color_llama, label="Llama3.1-LLM2Vec-8B", linestyle="None", markersize=8),
    plt.Line2D([0], [0], marker=marker_gmb, color=color_gmb, label="Count-based + GMB", linestyle="None", markersize=8),
]
# ax.legend(handles=legend_elements, loc="lower left", frameon=False, fontsize=10, title="Model Names")
# _plot_unified_legend(fig, np.array([[ax]]), fontsize=9)

# adjust layout for better spacing
# plt.tight_layout()

# save the figure
# plt.savefig("../../EHRSHOT_ASSETS/figures_noel/context_size_performance_adapted.png", dpi=300, bbox_inches='tight')
# plt.savefig("../../EHRSHOT_ASSETS/figures_noel/context_size_performance_adapted.pdf", dpi=300, bbox_inches='tight')

# plt.show()

## Plotting

In [ ]:
%matplotlib inline
_plot_unified_legend(fig, axes, fontsize=8)
fig.savefig('../../EHRSHOT_ASSETS/figures_noel/combined_context-time.pdf', dpi=300, bbox_inches='tight')
fig